In [1]:
import torch
import torch.nn as nn
torch.set_default_dtype(torch.float64)
from collections import OrderedDict
import math
import numpy as np
import sympy as sp
import seaborn as sns
from scipy.interpolate import griddata
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import pandas as pd
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
from scipy.stats import qmc
import scipy.io

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
class ChebyKANLayer(nn.Module):
    def __init__(self, input_dim, output_dim, degree):
        super(ChebyKANLayer, self).__init__()
        self.inputdim = input_dim
        self.outdim = output_dim
        self.degree = degree

        # 学习able的切比雪夫系数：使用float64
        self.cheby_coeffs = nn.Parameter(torch.empty(input_dim, output_dim, degree + 1))
        
        # Xavier初始化，保持float64
        nn.init.xavier_normal_(self.cheby_coeffs)

        # 预计算多项式阶数索引，使用float64以避免类型转换
        self.register_buffer("arange", torch.arange(0, degree + 1, 1))    

        # self.cheby_coeffs = nn.Parameter(torch.empty(input_dim, output_dim, degree + 1))
        # nn.init.normal_(self.cheby_coeffs, mean=0.0, std=1/(input_dim * (degree + 1)))

    def forward(self, x):
        # 确保输入是2D (batch_size, input_dim)，并转换为float64
        x = torch.reshape(x, (-1, self.inputdim))

        # 将输入归一化到切比雪夫多项式的定义域[-1, 1]
        x = torch.tanh(x)

        # 初始化切比雪夫多项式张量：显式使用float64
        cheby = torch.ones(x.shape[0], self.inputdim, self.degree + 1, 
                          device=x.device)
        
        # 设置一阶切比雪夫多项式：T_1(x) = x
        if self.degree > 0:
            cheby[:, :, 1] = x
        
        # 递归计算高阶切比雪夫多项式
        # 递推关系：T_n(x) = 2 * x * T_{n-1}(x) - T_{n-2}(x)
        for i in range(2, self.degree + 1):
            cheby[:, :, i] = 2 * x * cheby[:, :, i - 1].clone() - cheby[:, :, i - 2].clone()

        # 用学习到的系数对切比雪夫多项式进行加权求和
        y = torch.einsum('bid,iod->bo', cheby, self.cheby_coeffs)  
        y = y.view(-1, self.outdim)
        return y

class KAN(nn.Module):
    def __init__(self, layer_widths, degree, lb, ub):
        super(KAN, self).__init__()
        self.layer_widths = layer_widths
        self.degree = degree
        self.lb = lb
        self.ub = ub
        self.Cheb_layers = nn.ModuleList()
        for i in range(len(layer_widths) - 1):
            self.Cheb_layers.append(ChebyKANLayer(layer_widths[i], layer_widths[i+1], degree))
    
    def forward(self, x):
        out = 2 * (x - self.lb) / (self.ub - self.lb) - 1  # 归一化到[-1, 1]
        for layer in self.Cheb_layers:
            out = layer(out)
        return out   

In [4]:
# the physics-guided neural network
class PIKAN(nn.Module):
    def __init__(self, X_ub,X_lb,X_ic,X_res,u_ic,v_ic,alpha,beta,gamma,layers, degree, lb, ub):
        super(PIKAN, self).__init__()
        # boundary conditions
        
        self.z_ub = torch.tensor(X_ub[:, 0:1], requires_grad=True).double().to(device)
        self.t_ub = torch.tensor(X_ub[:, 1:2], requires_grad=True).double().to(device)

        self.z_lb = torch.tensor(X_lb[:, 0:1], requires_grad=True).double().to(device)
        self.t_lb = torch.tensor(X_lb[:, 1:2], requires_grad=True).double().to(device)

        self.z_ic = torch.tensor(X_ic[:, 0:1], requires_grad=True).double().to(device)
        self.t_ic = torch.tensor(X_ic[:, 1:2], requires_grad=True).double().to(device)
        self.u_ic = torch.tensor(u_ic).double().to(device)
        self.v_ic = torch.tensor(v_ic).double().to(device)
        self.u_ub_e = torch.tensor(u_ub_e).double().to(device)
        self.v_ub_e = torch.tensor(v_ub_e).double().to(device)
        self.ux_ub_e = torch.tensor(ux_ub_e).double().to(device)
        self.vx_ub_e = torch.tensor(vx_ub_e).double().to(device)
        self.u_lb_e = torch.tensor(u_lb_e).double().to(device)
        self.v_lb_e = torch.tensor(v_lb_e).double().to(device)
        self.ux_lb_e = torch.tensor(ux_lb_e).double().to(device)
        self.vx_lb_e = torch.tensor(vx_lb_e).double().to(device)

        self.z_res = torch.tensor(X_res[:, 0:1], requires_grad=True).double().to(device)
        self.t_res = torch.tensor(X_res[:, 1:2], requires_grad=True).double().to(device)

        self.alpha = torch.tensor(alpha).double().to(device)
        self.beta = torch.tensor(beta).double().to(device)
        self.gamma = torch.tensor(gamma).double().to(device)

        self.lb = torch.tensor(lb).double().to(device)  # 归一化下界
        self.ub = torch.tensor(ub).double().to(device)  # 归一

        self.layers = layers
        
        self.net = KAN(layers, degree, self.lb, self.ub).to(device)

        self.iter = 0

        self.optimizer1 = torch.optim.Adam(self.net.parameters(),lr=1e-3)

        self.optimizer2 = torch.optim.LBFGS(
            self.net.parameters(), 
            lr=1.0, #步长缩放因子，用于控制拟牛顿算法中搜索的步长
            max_iter=40000, #最大迭代次数
            max_eval=40000, #函数和梯度最大评估次数
            history_size=50, #存储和更新历史信息的缓存大小
            tolerance_grad=1e-7, #表示梯度的容差，即梯度的变化小于该容差时，算法将终止
            tolerance_change=1.0 * np.finfo(float).eps, #表示优化变化的容差，即参数变化量的绝对值小于该容差时，算法将终止
            line_search_fn="strong_wolfe" #can be "strong_wolfe"
        )

        self.loss_history = {
            'total': [],
            'residual': [],
            'ic': [],
            'b': []
        }
        
    def net_h(self, z, t):  
        H = self.net(torch.cat([z, t], dim=1))
        u = H[:, 0:1]
        v = H[:, 1:2]
        return u,v
        
    def gradient(self, y, x):
        return torch.autograd.grad(y, x, grad_outputs=torch.ones_like(y), retain_graph=True, create_graph=True)[0]
    
    def net_b(self, z, t):

        u,v = self.net_h(z, t)
        u_z = self.gradient(u, z)
        v_z = self.gradient(v, z)

        return u, v, u_z, v_z            
    
    def net_ic(self, z, t):

        u, v = self.net_h(z, t)

        return u, v      
    
    def net_f(self, z, t):

        u,v = self.net_h(z, t)
        v_t = self.gradient(v, t)
        v_z = self.gradient(v, z)

        u_t = self.gradient(u, t)
        u_z = self.gradient(u, z)
        u_zz = self.gradient(u_z, z)
        u_zzz = self.gradient(u_zz, z)

        f1 = u_t - v_z
        f2 = v_t + self.alpha * u_z + self.beta * (2*u*u_z) + self.gamma * u_zzz

        return f1, f2

    def loss_func(self):

        self.optimizer1.zero_grad()
        self.optimizer2.zero_grad()

        u_ic_pred, v_ic_pred = self.net_ic(self.z_ic, self.t_ic)

        u_ub_pred, v_ub_pred, u_z_ub_pred, v_z_ub_pred = self.net_b(self.z_ub, self.t_ub)

        u_lb_pred, v_lb_pred, u_z_lb_pred, v_z_lb_pred = self.net_b(self.z_lb, self.t_lb)

        f1_res_pred, f2_res_pred = self.net_f(self.z_res, self.t_res)

        loss_res1 = torch.mean(f1_res_pred**2)
        loss_res2 = torch.mean(f2_res_pred**2)
        loss_res = loss_res1 + loss_res2
        loss_ic1 = torch.mean((u_ic_pred - self.u_ic)**2)
        loss_ic2 = torch.mean((v_ic_pred - self.v_ic)**2)
        loss_b1 = torch.mean((u_ub_pred - self.u_ub_e)**2)
        loss_b2 = torch.mean((v_ub_pred - self.v_ub_e)**2)
        loss_b3 = torch.mean((u_z_ub_pred - self.ux_ub_e)**2)
        loss_b4 = torch.mean((v_z_ub_pred - self.vx_ub_e)**2)
        loss_b5 = torch.mean((u_lb_pred - self.u_lb_e)**2)
        loss_b6 = torch.mean((v_lb_pred - self.v_lb_e)**2)
        loss_b7 = torch.mean((u_z_lb_pred - self.ux_lb_e)**2)
        loss_b8 = torch.mean((v_z_lb_pred - self.vx_lb_e)**2)

        loss_ic = loss_ic1 + loss_ic2
        loss_b = loss_b1 + loss_b2 + loss_b3 + loss_b4 + loss_b5 + loss_b6 + loss_b7 + loss_b8

        loss = loss_res + loss_ic1 + loss_ic2 + loss_b

        # self.loss_history['total'].append(loss.item())
        # self.loss_history['residual'].append(loss_res.item())
        # self.loss_history['ic'].append(loss_ic.item())
        # self.loss_history['b'].append(loss_b.item())
        
        loss.backward()
        self.iter += 1
        if self.iter % 100 == 0:
            self.loss_history['total'].append(loss.item())
            self.loss_history['residual'].append(loss_res.item())
            self.loss_history['ic'].append(loss_ic.item())
            self.loss_history['b'].append(loss_b.item())
            print(
                'Iter %d, Loss: %.5e, Loss_res: %.5e, Loss_ic1: %.5e, Loss_ic2: %.5e, Loss_b1: %.5e, Loss_b2: %.5e, Loss_b3: %.5e, Loss_b4: %.5e' % 
                (   self.iter, 
                    loss.item(), 
                    loss_res.item(), 
                    loss_ic1.item(),
                    loss_ic2.item(), 
                    loss_b1.item(),
                    loss_b2.item(),
                    loss_b3.item(),
                    loss_b4.item()
                )
            )

        return loss
    
    def train(self):
        self.net.train()
        import time
        t_start = time.time()
        print("采用Adam优化器")
        for i in range(20000):
            self.optimizer1.step(self.loss_func)
        # 然后运行lbfgs优化器
        print("采用L-BFGS优化器")
        self.optimizer2.step(self.loss_func)
        print('Total training time: %.2f h (%.0f min)' % ((time.time()-t_start)/3600, (time.time()-t_start)/60))
 
            
    def predict(self, X):
        
        z = torch.tensor(X[:, 0:1], requires_grad=True).double().to(device)
        t = torch.tensor(X[:, 1:2], requires_grad=True).double().to(device)

        self.net.eval()
        u,v = self.net_h(z, t)
        u = u.detach().cpu().numpy()
        v = v.detach().cpu().numpy()

        return u, v

In [5]:
# reproducibility: random seed 1 (covers LHS sampling and network init)
import random
seed = 4
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# alpha = -1
# beta = -3
# gamma = -1

# N_res = 1000
# N_b = 100
# N_ic = 100

# x = np.linspace(-15, 15, 201)
# t = np.linspace(0, 5, 201) 
# x_mesh, t_mesh = np.meshgrid(x, t)

# X = np.hstack((x_mesh.flatten()[:,None], t_mesh.flatten()[:,None]))

# lb = X.min(0) #下界x'QxQ
# ub = X.max(0)  #上界 

# def lhs(n, size):
#     """生成n维拉丁超立方采样，返回size个样本点"""
#     sampler = qmc.LatinHypercube(d=n)
#     return sampler.random(n=size)

# # 使用简洁形式进行采样（与你要求的形式一致）
# X_res = lb + (ub - lb) * lhs(2, N_res)  # 2维残差点采样

# t_samples = lb[1] + (ub[1] - lb[1]) * lhs(1, N_b)  # 时间t的采样：[N_b, 1]

# # 上边界点（x=ub[0]，与下边界对称）
# X_ub = np.hstack((ub[0] * np.ones((N_b, 1)),t_samples))
# X_lb = np.hstack((lb[0] * np.ones((N_b, 1)),t_samples))

# X_ic = np.hstack((lb[0] + (ub[0] - lb[0]) * lhs(1, N_ic), 
#                   lb[1] * np.ones((N_ic, 1))))  # 初始条件点

In [6]:
alpha = -1
beta = -3
gamma = -1

x = np.linspace(-15, 15, 301)
t = np.linspace(0, 5, 201) 
x_mesh, t_mesh = np.meshgrid(x, t)

X = np.hstack((x_mesh.flatten()[:,None], t_mesh.flatten()[:,None]))

lb = X.min(0) #下界x'QxQ
ub = X.max(0)  #上界 

df_h = np.load(r"D:\Graduate\S\SU\PI-A-KMNet-for-Special-Nonlinear-equation-main\KD\PS-20260622\新实验\bq-seed\5000\KAN\residual_10000.npz")
X_res = df_h['X_res']
X_ub = df_h['X_ub']
X_lb = df_h['X_lb']
X_ic = df_h['X_ic']

In [7]:
def func_u(X):
    x, t = X[:, 0:1], X[:, 1:2]
    zeta = 1/6
    y1 = 0
    y2 = 0
    y3 = 2
    k1 = 1
    k2 = 1
    k3 = 5/4
    varsigma1 = -2
    varsigma2 = 4
    varsigma3 = -4

    u=-alpha/(2*beta) + zeta + 6*gamma*(-k1**2*np.exp(k1*t*np.sqrt(-2*beta*zeta - gamma*k1**2) + k1*(x + y1) + varsigma1) - k2**2*np.exp(-k2*t*np.sqrt(-2*beta*zeta - gamma*k2**2) + k2*(x + y2) + varsigma2) - k3**2*np.exp(k3*t*np.sqrt(-2*beta*zeta - gamma*k3**2) + k3*(x + y3) + varsigma3) + (k1 + k2)**2*(2*beta*zeta*(k1 - k2)**2 + gamma*(k1 - k2)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) + k2*np.sqrt(-2*beta*zeta - gamma*k2**2))**2)*np.exp(k1*t*np.sqrt(-2*beta*zeta - gamma*k1**2) + k1*(x + y1) - k2*t*np.sqrt(-2*beta*zeta - gamma*k2**2) + k2*(x + y2) + varsigma1 + varsigma2)/(2*beta*zeta*(k1 + k2)**2 + gamma*(k1 + k2)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) - k2*np.sqrt(-2*beta*zeta - gamma*k2**2))**2) + (k1 + k3)**2*(2*beta*zeta*(k1 - k3)**2 + gamma*(k1 - k3)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) - k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)*np.exp(k1*t*np.sqrt(-2*beta*zeta - gamma*k1**2) + k1*(x + y1) + k3*t*np.sqrt(-2*beta*zeta - gamma*k3**2) + k3*(x + y3) + varsigma1 + varsigma3)/(2*beta*zeta*(k1 + k3)**2 + gamma*(k1 + k3)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) + k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2) + (k2 + k3)**2*(2*beta*zeta*(k2 - k3)**2 + gamma*(k2 - k3)**4 + (k2*np.sqrt(-2*beta*zeta - gamma*k2**2) + k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)*np.exp(-k2*t*np.sqrt(-2*beta*zeta - gamma*k2**2) + k2*(x + y2) + k3*t*np.sqrt(-2*beta*zeta - gamma*k3**2) + k3*(x + y3) + varsigma2 + varsigma3)/(2*beta*zeta*(k2 + k3)**2 + gamma*(k2 + k3)**4 + (k2*np.sqrt(-2*beta*zeta - gamma*k2**2) - k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2) + (k1 + k2 + k3)**2*(2*beta*zeta*(k1 - k2)**2 + gamma*(k1 - k2)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) + k2*np.sqrt(-2*beta*zeta - gamma*k2**2))**2)*(2*beta*zeta*(k1 - k3)**2 + gamma*(k1 - k3)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) - k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)*(2*beta*zeta*(k2 - k3)**2 + gamma*(k2 - k3)**4 + (k2*np.sqrt(-2*beta*zeta - gamma*k2**2) + k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)*np.exp(k1*t*np.sqrt(-2*beta*zeta - gamma*k1**2) + k1*(x + y1) - k2*t*np.sqrt(-2*beta*zeta - gamma*k2**2) + k2*(x + y2) + k3*t*np.sqrt(-2*beta*zeta - gamma*k3**2) + k3*(x + y3) + varsigma1 + varsigma2 + varsigma3)/((2*beta*zeta*(k1 + k2)**2 + gamma*(k1 + k2)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) - k2*np.sqrt(-2*beta*zeta - gamma*k2**2))**2)*(2*beta*zeta*(k1 + k3)**2 + gamma*(k1 + k3)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) + k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)*(2*beta*zeta*(k2 + k3)**2 + gamma*(k2 + k3)**4 + (k2*np.sqrt(-2*beta*zeta - gamma*k2**2) - k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)) - (-k1*np.exp(k1*t*np.sqrt(-2*beta*zeta - gamma*k1**2) + k1*(x + y1) + varsigma1) - k2*np.exp(-k2*t*np.sqrt(-2*beta*zeta - gamma*k2**2) + k2*(x + y2) + varsigma2) - k3*np.exp(k3*t*np.sqrt(-2*beta*zeta - gamma*k3**2) + k3*(x + y3) + varsigma3) + (k1 + k2)*(2*beta*zeta*(k1 - k2)**2 + gamma*(k1 - k2)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) + k2*np.sqrt(-2*beta*zeta - gamma*k2**2))**2)*np.exp(k1*t*np.sqrt(-2*beta*zeta - gamma*k1**2) + k1*(x + y1) - k2*t*np.sqrt(-2*beta*zeta - gamma*k2**2) + k2*(x + y2) + varsigma1 + varsigma2)/(2*beta*zeta*(k1 + k2)**2 + gamma*(k1 + k2)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) - k2*np.sqrt(-2*beta*zeta - gamma*k2**2))**2) + (k1 + k3)*(2*beta*zeta*(k1 - k3)**2 + gamma*(k1 - k3)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) - k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)*np.exp(k1*t*np.sqrt(-2*beta*zeta - gamma*k1**2) + k1*(x + y1) + k3*t*np.sqrt(-2*beta*zeta - gamma*k3**2) + k3*(x + y3) + varsigma1 + varsigma3)/(2*beta*zeta*(k1 + k3)**2 + gamma*(k1 + k3)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) + k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2) + (k2 + k3)*(2*beta*zeta*(k2 - k3)**2 + gamma*(k2 - k3)**4 + (k2*np.sqrt(-2*beta*zeta - gamma*k2**2) + k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)*np.exp(-k2*t*np.sqrt(-2*beta*zeta - gamma*k2**2) + k2*(x + y2) + k3*t*np.sqrt(-2*beta*zeta - gamma*k3**2) + k3*(x + y3) + varsigma2 + varsigma3)/(2*beta*zeta*(k2 + k3)**2 + gamma*(k2 + k3)**4 + (k2*np.sqrt(-2*beta*zeta - gamma*k2**2) - k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2) + (k1 + k2 + k3)*(2*beta*zeta*(k1 - k2)**2 + gamma*(k1 - k2)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) + k2*np.sqrt(-2*beta*zeta - gamma*k2**2))**2)*(2*beta*zeta*(k1 - k3)**2 + gamma*(k1 - k3)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) - k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)*(2*beta*zeta*(k2 - k3)**2 + gamma*(k2 - k3)**4 + (k2*np.sqrt(-2*beta*zeta - gamma*k2**2) + k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)*np.exp(k1*t*np.sqrt(-2*beta*zeta - gamma*k1**2) + k1*(x + y1) - k2*t*np.sqrt(-2*beta*zeta - gamma*k2**2) + k2*(x + y2) + k3*t*np.sqrt(-2*beta*zeta - gamma*k3**2) + k3*(x + y3) + varsigma1 + varsigma2 + varsigma3)/((2*beta*zeta*(k1 + k2)**2 + gamma*(k1 + k2)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) - k2*np.sqrt(-2*beta*zeta - gamma*k2**2))**2)*(2*beta*zeta*(k1 + k3)**2 + gamma*(k1 + k3)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) + k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)*(2*beta*zeta*(k2 + k3)**2 + gamma*(k2 + k3)**4 + (k2*np.sqrt(-2*beta*zeta - gamma*k2**2) - k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)))**2/((2*beta*zeta*(k1 - k2)**2 + gamma*(k1 - k2)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) + k2*np.sqrt(-2*beta*zeta - gamma*k2**2))**2)*(2*beta*zeta*(k1 - k3)**2 + gamma*(k1 - k3)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) - k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)*(2*beta*zeta*(k2 - k3)**2 + gamma*(k2 - k3)**4 + (k2*np.sqrt(-2*beta*zeta - gamma*k2**2) + k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)*np.exp(k1*t*np.sqrt(-2*beta*zeta - gamma*k1**2) + k1*(x + y1) - k2*t*np.sqrt(-2*beta*zeta - gamma*k2**2) + k2*(x + y2) + k3*t*np.sqrt(-2*beta*zeta - gamma*k3**2) + k3*(x + y3) + varsigma1 + varsigma2 + varsigma3)/((2*beta*zeta*(k1 + k2)**2 + gamma*(k1 + k2)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) - k2*np.sqrt(-2*beta*zeta - gamma*k2**2))**2)*(2*beta*zeta*(k1 + k3)**2 + gamma*(k1 + k3)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) + k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)*(2*beta*zeta*(k2 + k3)**2 + gamma*(k2 + k3)**4 + (k2*np.sqrt(-2*beta*zeta - gamma*k2**2) - k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)) + (2*beta*zeta*(k1 - k2)**2 + gamma*(k1 - k2)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) + k2*np.sqrt(-2*beta*zeta - gamma*k2**2))**2)*np.exp(k1*t*np.sqrt(-2*beta*zeta - gamma*k1**2) + k1*(x + y1) - k2*t*np.sqrt(-2*beta*zeta - gamma*k2**2) + k2*(x + y2) + varsigma1 + varsigma2)/(2*beta*zeta*(k1 + k2)**2 + gamma*(k1 + k2)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) - k2*np.sqrt(-2*beta*zeta - gamma*k2**2))**2) + (2*beta*zeta*(k1 - k3)**2 + gamma*(k1 - k3)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) - k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)*np.exp(k1*t*np.sqrt(-2*beta*zeta - gamma*k1**2) + k1*(x + y1) + k3*t*np.sqrt(-2*beta*zeta - gamma*k3**2) + k3*(x + y3) + varsigma1 + varsigma3)/(2*beta*zeta*(k1 + k3)**2 + gamma*(k1 + k3)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) + k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2) + (2*beta*zeta*(k2 - k3)**2 + gamma*(k2 - k3)**4 + (k2*np.sqrt(-2*beta*zeta - gamma*k2**2) + k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)*np.exp(-k2*t*np.sqrt(-2*beta*zeta - gamma*k2**2) + k2*(x + y2) + k3*t*np.sqrt(-2*beta*zeta - gamma*k3**2) + k3*(x + y3) + varsigma2 + varsigma3)/(2*beta*zeta*(k2 + k3)**2 + gamma*(k2 + k3)**4 + (k2*np.sqrt(-2*beta*zeta - gamma*k2**2) - k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2) - np.exp(k1*t*np.sqrt(-2*beta*zeta - gamma*k1**2) + k1*(x + y1) + varsigma1) - np.exp(-k2*t*np.sqrt(-2*beta*zeta - gamma*k2**2) + k2*(x + y2) + varsigma2) - np.exp(k3*t*np.sqrt(-2*beta*zeta - gamma*k3**2) + k3*(x + y3) + varsigma3) - 1))/(beta*((2*beta*zeta*(k1 - k2)**2 + gamma*(k1 - k2)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) + k2*np.sqrt(-2*beta*zeta - gamma*k2**2))**2)*(2*beta*zeta*(k1 - k3)**2 + gamma*(k1 - k3)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) - k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)*(2*beta*zeta*(k2 - k3)**2 + gamma*(k2 - k3)**4 + (k2*np.sqrt(-2*beta*zeta - gamma*k2**2) + k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)*np.exp(k1*t*np.sqrt(-2*beta*zeta - gamma*k1**2) + k1*(x + y1) - k2*t*np.sqrt(-2*beta*zeta - gamma*k2**2) + k2*(x + y2) + k3*t*np.sqrt(-2*beta*zeta - gamma*k3**2) + k3*(x + y3) + varsigma1 + varsigma2 + varsigma3)/((2*beta*zeta*(k1 + k2)**2 + gamma*(k1 + k2)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) - k2*np.sqrt(-2*beta*zeta - gamma*k2**2))**2)*(2*beta*zeta*(k1 + k3)**2 + gamma*(k1 + k3)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) + k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)*(2*beta*zeta*(k2 + k3)**2 + gamma*(k2 + k3)**4 + (k2*np.sqrt(-2*beta*zeta - gamma*k2**2) - k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)) + (2*beta*zeta*(k1 - k2)**2 + gamma*(k1 - k2)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) + k2*np.sqrt(-2*beta*zeta - gamma*k2**2))**2)*np.exp(k1*t*np.sqrt(-2*beta*zeta - gamma*k1**2) + k1*(x + y1) - k2*t*np.sqrt(-2*beta*zeta - gamma*k2**2) + k2*(x + y2) + varsigma1 + varsigma2)/(2*beta*zeta*(k1 + k2)**2 + gamma*(k1 + k2)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) - k2*np.sqrt(-2*beta*zeta - gamma*k2**2))**2) + (2*beta*zeta*(k1 - k3)**2 + gamma*(k1 - k3)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) - k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)*np.exp(k1*t*np.sqrt(-2*beta*zeta - gamma*k1**2) + k1*(x + y1) + k3*t*np.sqrt(-2*beta*zeta - gamma*k3**2) + k3*(x + y3) + varsigma1 + varsigma3)/(2*beta*zeta*(k1 + k3)**2 + gamma*(k1 + k3)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) + k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2) + (2*beta*zeta*(k2 - k3)**2 + gamma*(k2 - k3)**4 + (k2*np.sqrt(-2*beta*zeta - gamma*k2**2) + k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)*np.exp(-k2*t*np.sqrt(-2*beta*zeta - gamma*k2**2) + k2*(x + y2) + k3*t*np.sqrt(-2*beta*zeta - gamma*k3**2) + k3*(x + y3) + varsigma2 + varsigma3)/(2*beta*zeta*(k2 + k3)**2 + gamma*(k2 + k3)**4 + (k2*np.sqrt(-2*beta*zeta - gamma*k2**2) - k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2) - np.exp(k1*t*np.sqrt(-2*beta*zeta - gamma*k1**2) + k1*(x + y1) + varsigma1) - np.exp(-k2*t*np.sqrt(-2*beta*zeta - gamma*k2**2) + k2*(x + y2) + varsigma2) - np.exp(k3*t*np.sqrt(-2*beta*zeta - gamma*k3**2) + k3*(x + y3) + varsigma3) - 1))
    return u

def func_v(X):
    x, t = X[:, 0:1], X[:, 1:2]
    zeta = 1/6
    y1 = 0
    y2 = 0
    y3 = 2
    k1 = 1
    k2 = 1
    k3 = 5/4
    varsigma1 = -2
    varsigma2 = 4
    varsigma3 = -4

    v=6*gamma*(-k1**2*np.sqrt(-2*beta*zeta - gamma*k1**2)*np.exp(k1*t*np.sqrt(-2*beta*zeta - gamma*k1**2) + k1*(x + y1) + varsigma1) + k2**2*np.sqrt(-2*beta*zeta - gamma*k2**2)*np.exp(-k2*t*np.sqrt(-2*beta*zeta - gamma*k2**2) + k2*(x + y2) + varsigma2) - k3**2*np.sqrt(-2*beta*zeta - gamma*k3**2)*np.exp(k3*t*np.sqrt(-2*beta*zeta - gamma*k3**2) + k3*(x + y3) + varsigma3) + (k1 + k2)*(k1*np.sqrt(-2*beta*zeta - gamma*k1**2) - k2*np.sqrt(-2*beta*zeta - gamma*k2**2))*(2*beta*zeta*(k1 - k2)**2 + gamma*(k1 - k2)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) + k2*np.sqrt(-2*beta*zeta - gamma*k2**2))**2)*np.exp(k1*t*np.sqrt(-2*beta*zeta - gamma*k1**2) + k1*(x + y1) - k2*t*np.sqrt(-2*beta*zeta - gamma*k2**2) + k2*(x + y2) + varsigma1 + varsigma2)/(2*beta*zeta*(k1 + k2)**2 + gamma*(k1 + k2)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) - k2*np.sqrt(-2*beta*zeta - gamma*k2**2))**2) + (k1 + k3)*(k1*np.sqrt(-2*beta*zeta - gamma*k1**2) + k3*np.sqrt(-2*beta*zeta - gamma*k3**2))*(2*beta*zeta*(k1 - k3)**2 + gamma*(k1 - k3)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) - k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)*np.exp(k1*t*np.sqrt(-2*beta*zeta - gamma*k1**2) + k1*(x + y1) + k3*t*np.sqrt(-2*beta*zeta - gamma*k3**2) + k3*(x + y3) + varsigma1 + varsigma3)/(2*beta*zeta*(k1 + k3)**2 + gamma*(k1 + k3)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) + k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2) - (k2 + k3)*(k2*np.sqrt(-2*beta*zeta - gamma*k2**2) - k3*np.sqrt(-2*beta*zeta - gamma*k3**2))*(2*beta*zeta*(k2 - k3)**2 + gamma*(k2 - k3)**4 + (k2*np.sqrt(-2*beta*zeta - gamma*k2**2) + k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)*np.exp(-k2*t*np.sqrt(-2*beta*zeta - gamma*k2**2) + k2*(x + y2) + k3*t*np.sqrt(-2*beta*zeta - gamma*k3**2) + k3*(x + y3) + varsigma2 + varsigma3)/(2*beta*zeta*(k2 + k3)**2 + gamma*(k2 + k3)**4 + (k2*np.sqrt(-2*beta*zeta - gamma*k2**2) - k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2) + (k1 + k2 + k3)*(k1*np.sqrt(-2*beta*zeta - gamma*k1**2) - k2*np.sqrt(-2*beta*zeta - gamma*k2**2) + k3*np.sqrt(-2*beta*zeta - gamma*k3**2))*(2*beta*zeta*(k1 - k2)**2 + gamma*(k1 - k2)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) + k2*np.sqrt(-2*beta*zeta - gamma*k2**2))**2)*(2*beta*zeta*(k1 - k3)**2 + gamma*(k1 - k3)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) - k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)*(2*beta*zeta*(k2 - k3)**2 + gamma*(k2 - k3)**4 + (k2*np.sqrt(-2*beta*zeta - gamma*k2**2) + k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)*np.exp(k1*t*np.sqrt(-2*beta*zeta - gamma*k1**2) + k1*(x + y1) - k2*t*np.sqrt(-2*beta*zeta - gamma*k2**2) + k2*(x + y2) + k3*t*np.sqrt(-2*beta*zeta - gamma*k3**2) + k3*(x + y3) + varsigma1 + varsigma2 + varsigma3)/((2*beta*zeta*(k1 + k2)**2 + gamma*(k1 + k2)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) - k2*np.sqrt(-2*beta*zeta - gamma*k2**2))**2)*(2*beta*zeta*(k1 + k3)**2 + gamma*(k1 + k3)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) + k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)*(2*beta*zeta*(k2 + k3)**2 + gamma*(k2 + k3)**4 + (k2*np.sqrt(-2*beta*zeta - gamma*k2**2) - k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)) - (-k1*np.exp(k1*t*np.sqrt(-2*beta*zeta - gamma*k1**2) + k1*(x + y1) + varsigma1) - k2*np.exp(-k2*t*np.sqrt(-2*beta*zeta - gamma*k2**2) + k2*(x + y2) + varsigma2) - k3*np.exp(k3*t*np.sqrt(-2*beta*zeta - gamma*k3**2) + k3*(x + y3) + varsigma3) + (k1 + k2)*(2*beta*zeta*(k1 - k2)**2 + gamma*(k1 - k2)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) + k2*np.sqrt(-2*beta*zeta - gamma*k2**2))**2)*np.exp(k1*t*np.sqrt(-2*beta*zeta - gamma*k1**2) + k1*(x + y1) - k2*t*np.sqrt(-2*beta*zeta - gamma*k2**2) + k2*(x + y2) + varsigma1 + varsigma2)/(2*beta*zeta*(k1 + k2)**2 + gamma*(k1 + k2)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) - k2*np.sqrt(-2*beta*zeta - gamma*k2**2))**2) + (k1 + k3)*(2*beta*zeta*(k1 - k3)**2 + gamma*(k1 - k3)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) - k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)*np.exp(k1*t*np.sqrt(-2*beta*zeta - gamma*k1**2) + k1*(x + y1) + k3*t*np.sqrt(-2*beta*zeta - gamma*k3**2) + k3*(x + y3) + varsigma1 + varsigma3)/(2*beta*zeta*(k1 + k3)**2 + gamma*(k1 + k3)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) + k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2) + (k2 + k3)*(2*beta*zeta*(k2 - k3)**2 + gamma*(k2 - k3)**4 + (k2*np.sqrt(-2*beta*zeta - gamma*k2**2) + k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)*np.exp(-k2*t*np.sqrt(-2*beta*zeta - gamma*k2**2) + k2*(x + y2) + k3*t*np.sqrt(-2*beta*zeta - gamma*k3**2) + k3*(x + y3) + varsigma2 + varsigma3)/(2*beta*zeta*(k2 + k3)**2 + gamma*(k2 + k3)**4 + (k2*np.sqrt(-2*beta*zeta - gamma*k2**2) - k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2) + (k1 + k2 + k3)*(2*beta*zeta*(k1 - k2)**2 + gamma*(k1 - k2)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) + k2*np.sqrt(-2*beta*zeta - gamma*k2**2))**2)*(2*beta*zeta*(k1 - k3)**2 + gamma*(k1 - k3)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) - k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)*(2*beta*zeta*(k2 - k3)**2 + gamma*(k2 - k3)**4 + (k2*np.sqrt(-2*beta*zeta - gamma*k2**2) + k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)*np.exp(k1*t*np.sqrt(-2*beta*zeta - gamma*k1**2) + k1*(x + y1) - k2*t*np.sqrt(-2*beta*zeta - gamma*k2**2) + k2*(x + y2) + k3*t*np.sqrt(-2*beta*zeta - gamma*k3**2) + k3*(x + y3) + varsigma1 + varsigma2 + varsigma3)/((2*beta*zeta*(k1 + k2)**2 + gamma*(k1 + k2)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) - k2*np.sqrt(-2*beta*zeta - gamma*k2**2))**2)*(2*beta*zeta*(k1 + k3)**2 + gamma*(k1 + k3)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) + k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)*(2*beta*zeta*(k2 + k3)**2 + gamma*(k2 + k3)**4 + (k2*np.sqrt(-2*beta*zeta - gamma*k2**2) - k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)))*(-k1*np.sqrt(-2*beta*zeta - gamma*k1**2)*np.exp(k1*t*np.sqrt(-2*beta*zeta - gamma*k1**2) + k1*(x + y1) + varsigma1) + k2*np.sqrt(-2*beta*zeta - gamma*k2**2)*np.exp(-k2*t*np.sqrt(-2*beta*zeta - gamma*k2**2) + k2*(x + y2) + varsigma2) - k3*np.sqrt(-2*beta*zeta - gamma*k3**2)*np.exp(k3*t*np.sqrt(-2*beta*zeta - gamma*k3**2) + k3*(x + y3) + varsigma3) + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) - k2*np.sqrt(-2*beta*zeta - gamma*k2**2))*(2*beta*zeta*(k1 - k2)**2 + gamma*(k1 - k2)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) + k2*np.sqrt(-2*beta*zeta - gamma*k2**2))**2)*np.exp(k1*t*np.sqrt(-2*beta*zeta - gamma*k1**2) + k1*(x + y1) - k2*t*np.sqrt(-2*beta*zeta - gamma*k2**2) + k2*(x + y2) + varsigma1 + varsigma2)/(2*beta*zeta*(k1 + k2)**2 + gamma*(k1 + k2)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) - k2*np.sqrt(-2*beta*zeta - gamma*k2**2))**2) + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) + k3*np.sqrt(-2*beta*zeta - gamma*k3**2))*(2*beta*zeta*(k1 - k3)**2 + gamma*(k1 - k3)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) - k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)*np.exp(k1*t*np.sqrt(-2*beta*zeta - gamma*k1**2) + k1*(x + y1) + k3*t*np.sqrt(-2*beta*zeta - gamma*k3**2) + k3*(x + y3) + varsigma1 + varsigma3)/(2*beta*zeta*(k1 + k3)**2 + gamma*(k1 + k3)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) + k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2) - (k2*np.sqrt(-2*beta*zeta - gamma*k2**2) - k3*np.sqrt(-2*beta*zeta - gamma*k3**2))*(2*beta*zeta*(k2 - k3)**2 + gamma*(k2 - k3)**4 + (k2*np.sqrt(-2*beta*zeta - gamma*k2**2) + k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)*np.exp(-k2*t*np.sqrt(-2*beta*zeta - gamma*k2**2) + k2*(x + y2) + k3*t*np.sqrt(-2*beta*zeta - gamma*k3**2) + k3*(x + y3) + varsigma2 + varsigma3)/(2*beta*zeta*(k2 + k3)**2 + gamma*(k2 + k3)**4 + (k2*np.sqrt(-2*beta*zeta - gamma*k2**2) - k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2) + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) - k2*np.sqrt(-2*beta*zeta - gamma*k2**2) + k3*np.sqrt(-2*beta*zeta - gamma*k3**2))*(2*beta*zeta*(k1 - k2)**2 + gamma*(k1 - k2)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) + k2*np.sqrt(-2*beta*zeta - gamma*k2**2))**2)*(2*beta*zeta*(k1 - k3)**2 + gamma*(k1 - k3)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) - k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)*(2*beta*zeta*(k2 - k3)**2 + gamma*(k2 - k3)**4 + (k2*np.sqrt(-2*beta*zeta - gamma*k2**2) + k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)*np.exp(k1*t*np.sqrt(-2*beta*zeta - gamma*k1**2) + k1*(x + y1) - k2*t*np.sqrt(-2*beta*zeta - gamma*k2**2) + k2*(x + y2) + k3*t*np.sqrt(-2*beta*zeta - gamma*k3**2) + k3*(x + y3) + varsigma1 + varsigma2 + varsigma3)/((2*beta*zeta*(k1 + k2)**2 + gamma*(k1 + k2)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) - k2*np.sqrt(-2*beta*zeta - gamma*k2**2))**2)*(2*beta*zeta*(k1 + k3)**2 + gamma*(k1 + k3)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) + k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)*(2*beta*zeta*(k2 + k3)**2 + gamma*(k2 + k3)**4 + (k2*np.sqrt(-2*beta*zeta - gamma*k2**2) - k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)))/((2*beta*zeta*(k1 - k2)**2 + gamma*(k1 - k2)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) + k2*np.sqrt(-2*beta*zeta - gamma*k2**2))**2)*(2*beta*zeta*(k1 - k3)**2 + gamma*(k1 - k3)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) - k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)*(2*beta*zeta*(k2 - k3)**2 + gamma*(k2 - k3)**4 + (k2*np.sqrt(-2*beta*zeta - gamma*k2**2) + k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)*np.exp(k1*t*np.sqrt(-2*beta*zeta - gamma*k1**2) + k1*(x + y1) - k2*t*np.sqrt(-2*beta*zeta - gamma*k2**2) + k2*(x + y2) + k3*t*np.sqrt(-2*beta*zeta - gamma*k3**2) + k3*(x + y3) + varsigma1 + varsigma2 + varsigma3)/((2*beta*zeta*(k1 + k2)**2 + gamma*(k1 + k2)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) - k2*np.sqrt(-2*beta*zeta - gamma*k2**2))**2)*(2*beta*zeta*(k1 + k3)**2 + gamma*(k1 + k3)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) + k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)*(2*beta*zeta*(k2 + k3)**2 + gamma*(k2 + k3)**4 + (k2*np.sqrt(-2*beta*zeta - gamma*k2**2) - k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)) + (2*beta*zeta*(k1 - k2)**2 + gamma*(k1 - k2)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) + k2*np.sqrt(-2*beta*zeta - gamma*k2**2))**2)*np.exp(k1*t*np.sqrt(-2*beta*zeta - gamma*k1**2) + k1*(x + y1) - k2*t*np.sqrt(-2*beta*zeta - gamma*k2**2) + k2*(x + y2) + varsigma1 + varsigma2)/(2*beta*zeta*(k1 + k2)**2 + gamma*(k1 + k2)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) - k2*np.sqrt(-2*beta*zeta - gamma*k2**2))**2) + (2*beta*zeta*(k1 - k3)**2 + gamma*(k1 - k3)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) - k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)*np.exp(k1*t*np.sqrt(-2*beta*zeta - gamma*k1**2) + k1*(x + y1) + k3*t*np.sqrt(-2*beta*zeta - gamma*k3**2) + k3*(x + y3) + varsigma1 + varsigma3)/(2*beta*zeta*(k1 + k3)**2 + gamma*(k1 + k3)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) + k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2) + (2*beta*zeta*(k2 - k3)**2 + gamma*(k2 - k3)**4 + (k2*np.sqrt(-2*beta*zeta - gamma*k2**2) + k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)*np.exp(-k2*t*np.sqrt(-2*beta*zeta - gamma*k2**2) + k2*(x + y2) + k3*t*np.sqrt(-2*beta*zeta - gamma*k3**2) + k3*(x + y3) + varsigma2 + varsigma3)/(2*beta*zeta*(k2 + k3)**2 + gamma*(k2 + k3)**4 + (k2*np.sqrt(-2*beta*zeta - gamma*k2**2) - k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2) - np.exp(k1*t*np.sqrt(-2*beta*zeta - gamma*k1**2) + k1*(x + y1) + varsigma1) - np.exp(-k2*t*np.sqrt(-2*beta*zeta - gamma*k2**2) + k2*(x + y2) + varsigma2) - np.exp(k3*t*np.sqrt(-2*beta*zeta - gamma*k3**2) + k3*(x + y3) + varsigma3) - 1))/(beta*((2*beta*zeta*(k1 - k2)**2 + gamma*(k1 - k2)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) + k2*np.sqrt(-2*beta*zeta - gamma*k2**2))**2)*(2*beta*zeta*(k1 - k3)**2 + gamma*(k1 - k3)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) - k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)*(2*beta*zeta*(k2 - k3)**2 + gamma*(k2 - k3)**4 + (k2*np.sqrt(-2*beta*zeta - gamma*k2**2) + k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)*np.exp(k1*t*np.sqrt(-2*beta*zeta - gamma*k1**2) + k1*(x + y1) - k2*t*np.sqrt(-2*beta*zeta - gamma*k2**2) + k2*(x + y2) + k3*t*np.sqrt(-2*beta*zeta - gamma*k3**2) + k3*(x + y3) + varsigma1 + varsigma2 + varsigma3)/((2*beta*zeta*(k1 + k2)**2 + gamma*(k1 + k2)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) - k2*np.sqrt(-2*beta*zeta - gamma*k2**2))**2)*(2*beta*zeta*(k1 + k3)**2 + gamma*(k1 + k3)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) + k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)*(2*beta*zeta*(k2 + k3)**2 + gamma*(k2 + k3)**4 + (k2*np.sqrt(-2*beta*zeta - gamma*k2**2) - k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)) + (2*beta*zeta*(k1 - k2)**2 + gamma*(k1 - k2)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) + k2*np.sqrt(-2*beta*zeta - gamma*k2**2))**2)*np.exp(k1*t*np.sqrt(-2*beta*zeta - gamma*k1**2) + k1*(x + y1) - k2*t*np.sqrt(-2*beta*zeta - gamma*k2**2) + k2*(x + y2) + varsigma1 + varsigma2)/(2*beta*zeta*(k1 + k2)**2 + gamma*(k1 + k2)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) - k2*np.sqrt(-2*beta*zeta - gamma*k2**2))**2) + (2*beta*zeta*(k1 - k3)**2 + gamma*(k1 - k3)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) - k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)*np.exp(k1*t*np.sqrt(-2*beta*zeta - gamma*k1**2) + k1*(x + y1) + k3*t*np.sqrt(-2*beta*zeta - gamma*k3**2) + k3*(x + y3) + varsigma1 + varsigma3)/(2*beta*zeta*(k1 + k3)**2 + gamma*(k1 + k3)**4 + (k1*np.sqrt(-2*beta*zeta - gamma*k1**2) + k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2) + (2*beta*zeta*(k2 - k3)**2 + gamma*(k2 - k3)**4 + (k2*np.sqrt(-2*beta*zeta - gamma*k2**2) + k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2)*np.exp(-k2*t*np.sqrt(-2*beta*zeta - gamma*k2**2) + k2*(x + y2) + k3*t*np.sqrt(-2*beta*zeta - gamma*k3**2) + k3*(x + y3) + varsigma2 + varsigma3)/(2*beta*zeta*(k2 + k3)**2 + gamma*(k2 + k3)**4 + (k2*np.sqrt(-2*beta*zeta - gamma*k2**2) - k3*np.sqrt(-2*beta*zeta - gamma*k3**2))**2) - np.exp(k1*t*np.sqrt(-2*beta*zeta - gamma*k1**2) + k1*(x + y1) + varsigma1) - np.exp(-k2*t*np.sqrt(-2*beta*zeta - gamma*k2**2) + k2*(x + y2) + varsigma2) - np.exp(k3*t*np.sqrt(-2*beta*zeta - gamma*k3**2) + k3*(x + y3) + varsigma3) - 1))
    return v

u_ic = func_u(X_ic)
v_ic = func_v(X_ic)

# Dirichlet+Neumann boundary data from the exact solution (replaces periodic identification)
eps_b = 1e-5
u_ub_e  = func_u(X_ub);  v_ub_e  = func_v(X_ub)
u_lb_e  = func_u(X_lb);  v_lb_e  = func_v(X_lb)
ux_ub_e = (func_u(X_ub + [eps_b, 0]) - func_u(X_ub - [eps_b, 0])) / (2*eps_b)
vx_ub_e = (func_v(X_ub + [eps_b, 0]) - func_v(X_ub - [eps_b, 0])) / (2*eps_b)
ux_lb_e = (func_u(X_lb + [eps_b, 0]) - func_u(X_lb - [eps_b, 0])) / (2*eps_b)
vx_lb_e = (func_v(X_lb + [eps_b, 0]) - func_v(X_lb - [eps_b, 0])) / (2*eps_b)


In [8]:
degree = 4
layers = [2,14,14,14,14,14,14,2]

model = PIKAN(X_ub,X_lb,X_ic,X_res,u_ic,v_ic,alpha,beta,gamma,layers, degree, lb, ub)

In [9]:
def count_parameters(model):
    """计算模型中可训练参数的总个数"""
    total_params = 0
    for param in model.parameters():
        # 累加每个参数张量的元素数量
        total_params += param.numel()
    return total_params

param_count = count_parameters(model)
print(f"模型可训练参数总个数: {param_count:,}")  # 使用逗号作为千位分隔符

模型可训练参数总个数: 5,180


In [10]:
model.train()

采用Adam优化器


Iter 100, Loss: 2.73991e-01, Loss_res: 1.76891e-01, Loss_ic1: 2.23854e-02, Loss_ic2: 4.61955e-02, Loss_b1: 1.01647e-02, Loss_b2: 4.60601e-03, Loss_b3: 2.23443e-03, Loss_b4: 1.59532e-03


Iter 200, Loss: 1.43304e-01, Loss_res: 9.40463e-02, Loss_ic1: 1.08703e-02, Loss_ic2: 2.58573e-02, Loss_b1: 3.48048e-03, Loss_b2: 2.36551e-03, Loss_b3: 1.57807e-03, Loss_b4: 9.94343e-04


Iter 300, Loss: 9.84438e-02, Loss_res: 6.60953e-02, Loss_ic1: 7.70801e-03, Loss_ic2: 1.66849e-02, Loss_b1: 1.81348e-03, Loss_b2: 1.23567e-03, Loss_b3: 1.20300e-03, Loss_b4: 6.91515e-04


Iter 400, Loss: 7.40771e-02, Loss_res: 5.10622e-02, Loss_ic1: 5.75937e-03, Loss_ic2: 1.15516e-02, Loss_b1: 1.15628e-03, Loss_b2: 6.58052e-04, Loss_b3: 9.48136e-04, Loss_b4: 4.73500e-04


Iter 500, Loss: 5.85856e-02, Loss_res: 4.15567e-02, Loss_ic1: 4.43349e-03, Loss_ic2: 8.16890e-03, Loss_b1: 8.30010e-04, Loss_b2: 4.15623e-04, Loss_b3: 7.95442e-04, Loss_b4: 3.50740e-04


... (592 outputs left out)

Iter 59800, Loss: 3.17447e-07, Loss_res: 2.96076e-07, Loss_ic1: 2.96675e-09, Loss_ic2: 6.05224e-09, Loss_b1: 8.63560e-10, Loss_b2: 6.34237e-10, Loss_b3: 1.32858e-09, Loss_b4: 2.27616e-09


Iter 59900, Loss: 3.17430e-07, Loss_res: 2.96061e-07, Loss_ic1: 2.96623e-09, Loss_ic2: 6.05153e-09, Loss_b1: 8.63381e-10, Loss_b2: 6.34125e-10, Loss_b3: 1.32830e-09, Loss_b4: 2.27589e-09


Iter 60000, Loss: 3.17412e-07, Loss_res: 2.96046e-07, Loss_ic1: 2.96571e-09, Loss_ic2: 6.05083e-09, Loss_b1: 8.63202e-10, Loss_b2: 6.34013e-10, Loss_b3: 1.32803e-09, Loss_b4: 2.27561e-09


Total training time: 17.25 h (1035 min)


In [11]:
all_states1 = {"model": model.net.state_dict(), "LBFGS": model.optimizer2.state_dict()}
torch.save(obj=all_states1, f=r"D:\Graduate\S\SU\PI-A-KMNet-for-Special-Nonlinear-equation-main\KD\PS-20260622\新实验\bq-seed\5000\KAN\4\all_states1.pth")

In [12]:
u_p,v_p = model.predict(X)
u_true = func_u(X)
v_true = func_v(X)

In [13]:
L2 = np.linalg.norm(u_p.flatten() - u_true.flatten())/np.linalg.norm(u_true.flatten())

print (L2)

0.0021220456425035446


In [14]:
L2 = np.linalg.norm(v_p.flatten() - v_true.flatten())/np.linalg.norm(v_true.flatten())

print (L2)

0.0014411397152509502


In [15]:
pikandata = {
    'x': x_mesh.flatten()[:, None],       # x网格数据（竖排）
    't': t_mesh.flatten()[:, None],       # t网格数据（竖排）
    'h_true': u_true.flatten()[:, None],  # u真实值（竖排）
    'g_true': v_true.flatten()[:, None],  # v真实值（竖排）
    'h_pred': u_p.flatten()[:, None],     # u预测值（竖排）
    'g_pred': v_p.flatten()[:, None]      # v预测值（竖排）
}

# 保存到单个MAT文件
scipy.io.savemat(r"D:\Graduate\S\SU\PI-A-KMNet-for-Special-Nonlinear-equation-main\KD\PS-20260622\新实验\bq-seed\5000\KAN\4\all_data.mat", pikandata)

In [16]:
# scipy.io.savemat('3wavedata_PIKAN/x.mat', {'array': x_mesh.flatten()[:, None]})
# scipy.io.savemat('3wavedata_PIKAN/t.mat', {'array': t_mesh.flatten()[:, None]})
# scipy.io.savemat('3wavedata_PIKAN/h_true.mat', {'array': u_true.flatten()[:, None]})
# scipy.io.savemat('3wavedata_PIKAN/g_true.mat', {'array': v_true.flatten()[:, None]})
# scipy.io.savemat('3wavedata_PIKAN/h_pred.mat', {'array': u_p.flatten()[:, None]})
# scipy.io.savemat('3wavedata_PIKAN/g_pred.mat', {'array': v_p.flatten()[:, None]})

In [17]:
loss_total = model.loss_history['total']
loss_residual = model.loss_history['residual']
loss_ic = model.loss_history['ic']
loss_b = model.loss_history['b']

In [18]:
num_iterations = len(loss_total)  # 损失记录的总数量（即总迭代次数）
iter_steps = list(range(100, 100 + num_iterations * 100, 100))

In [19]:
# 3. 将数据转换为竖排（列向量）：通过reshape(-1, 1)实现，形状变为(n, 1)
iter_steps_col = np.array(iter_steps).reshape(-1, 1)       # 竖排迭代步数
loss_total_col = np.array(loss_total).reshape(-1, 1)       # 竖排总损失
loss_residual_col = np.array(loss_residual).reshape(-1, 1) # 竖排残差损失
loss_ic_col = np.array(loss_ic).reshape(-1, 1)             # 竖排初始条件损失
loss_b_col = np.array(loss_b).reshape(-1, 1)               # 竖排边界条件损失

# 4. 整理并保存为mat文件（此时所有数据均为列向量）
data_loss = {
    'iter_steps': iter_steps_col,
    'loss_total': loss_total_col,
    'loss_residual': loss_residual_col,
    'loss_ic': loss_ic_col,
    'loss_b': loss_b_col
}

scipy.io.savemat(r"D:\Graduate\S\SU\PI-A-KMNet-for-Special-Nonlinear-equation-main\KD\PS-20260622\新实验\bq-seed\5000\KAN\4\loss_history.mat", data_loss)